In [1]:
# Celula 1 - Imports
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.diabetes.train import train_diabetes_models
from src.diabetes.predict import DiabetesPredictor

sns.set_theme(style='whitegrid')
print('Imports carregados')

C:\Users\ricoi\POSTECH\tech-challenge-fase1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports carregados


In [2]:
# Celula 2 - Treinar modelos (salva em models/diabetes/)
modelos, resultados, scaler, features = train_diabetes_models()
print('Treino concluido!')

C:\Users\ricoi\POSTECH\tech-challenge-fase1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


2026/08/07 15:27:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registered model 'diabetes_prediction_logistic_regression' already exists. Creating a new version of this model...
Created version '5' of model 'diabetes_prediction_logistic_regression'.
2026/08/07 15:27:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run registrada: 4f3e3aedee124499a120475648881ebd


Registered model 'diabetes_prediction_decision_tree' already exists. Creating a new version of this model...
Created version '5' of model 'diabetes_prediction_decision_tree'.
2026/08/07 15:28:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

Registered model 'diabetes_prediction_gradient_boosting' already exists. Creating a new version of this model...
Created version '5' of model 'diabetes_prediction_gradient_boosting'.
2026/08/07 15:28:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run registrada: 3e1f77be065a4f149e2ed55d5a4083a8


Registered model 'diabetes_prediction_svm' already exists. Creating a new version of this model...
Created version '5' of model 'diabetes_prediction_svm'.
2026/08/07 15:29:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run registrada: 705c25a742954efc9a62ae1ae672e8ae


⚠️  Aviso ao registrar knn: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.metrics._dist_metrics.EuclideanDistance64', 'sklearn.neighbors._kd_tree.KDTree'].
   Continuando... Modelo salvo como .joblib, MLflow com metadados apenas.
Treino concluido!


In [3]:
# Celula 3 - Tabela de metricas por modelo
df_resultados = pd.DataFrame(resultados).T
df_resultados = df_resultados.round(4) * 100
df_resultados.columns = ['Acuracia', 'AUC', 'Recall', 'Precisao', 'F1']
df_resultados.index.name = 'Modelo'
print('Metricas (%) por modelo:')
df_resultados

Metricas (%) por modelo:


,Acuracia,AUC,Recall,Precisao,F1
Modelo,,,,,
logistic_regression,71.43,82.30,51.85,60.87,56.00
decision_tree,77.27,74.52,64.81,68.63,66.67
random_forest,72.73,83.46,70.37,59.38,64.41
gradient_boosting,73.38,79.43,57.41,63.27,60.19
svm,75.32,79.24,61.11,66.00,63.46
knn,70.13,74.05,51.85,58.33,54.90


In [4]:
# Celula 4 - Grafico comparativo
df_resultados.plot(kind='bar', figsize=(12, 6))
plt.title('Comparativo de Metricas - Diabetes', fontsize=14, fontweight='bold')
plt.ylabel('Score (%)')
plt.xlabel('Modelo')
plt.ylim(50, 100)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Tambem salvo em outputs/diabetes/comparativo.png
os.makedirs('outputs/diabetes', exist_ok=True)
plt.savefig('outputs/diabetes/comparativo_nb.png', dpi=150, bbox_inches='tight')
print('Grafico salvo em outputs/diabetes/comparativo_nb.png')

C:\Users\ricoi\AppData\Local\Temp\ipykernel_21192\849372255.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Grafico salvo em outputs/diabetes/comparativo_nb.png


In [5]:
# Celula 5 - Melhor modelo e predicao de exemplo
melhor = df_resultados['F1'].idxmax()
print(f'Melhor modelo por F1: {melhor} ({df_resultados.loc[melhor, "F1"]:.2f}%)')

# Exemplo: primeira paciente do dataset Pima (resultado real: diabetes)
predictor = DiabetesPredictor()
amostra = [6.0, 148.0, 72.0, 35.0, 0.0, 33.6, 0.627, 50.0]
result = predictor.predict(amostra, melhor)
print('\nPredicao de exemplo:')
for k, v in result.items():
    if k != 'features':
        print(f'  {k}: {v}')

Melhor modelo por F1: decision_tree (66.67%)

Predicao de exemplo:
  prediction: Positive
  probability_positive: 0.5604395604395604
  probability_negative: 0.43956043956043955
  model_used: decision_tree


In [6]:
# Celula 6 - Feature importance (Random Forest)
rf = modelos['random_forest']
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
n_top = min(15, len(features))  # diabetes tem apenas 8 features

plt.figure(figsize=(10, 8))
plt.barh(range(n_top), importances[indices[:n_top]][::-1], color='steelblue')
plt.yticks(range(n_top), [features[i] for i in indices[:n_top]][::-1])
plt.title(f'Feature Importance - Random Forest (Top {n_top})', fontsize=14, fontweight='bold')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

print(f'Top {min(10, len(features))} features mais importantes:')
for i in range(min(10, len(features))):
    print(f'  {i+1}. {features[indices[i]]}: {importances[indices[i]]:.4f}') 

Top 8 features mais importantes:
  1. glucose: 0.2730
  2. bmi: 0.1736
  3. age: 0.1308
  4. diabetes_pedigree: 0.1200
  5. blood_pressure: 0.0802
  6. pregnancies: 0.0786
  7. insulin: 0.0729
  8. skin_thickness: 0.0710


C:\Users\ricoi\AppData\Local\Temp\ipykernel_21192\2975048188.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Celula 7 - Explicabilidade SHAP (modelo de referencia, como no notebook FIAP)
try:
    import shap
    from src.diabetes.dataset import load_diabetes_data

    modelo_referencia = modelos["logistic_regression"]
    # Mesmo pre-processamento do treino (StandardScaler + split 80/20, seed 42)
    X_train_scaled, X_test_scaled, _, _, _, _ = load_diabetes_data(scale=True)

    background = X_train_scaled[:100]
    explainer = shap.Explainer(modelo_referencia, background, feature_names=features)
    shap_values = explainer(X_test_scaled[:50])

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_scaled[:50], feature_names=features, show=False)
    plt.tight_layout()
    plt.show()
    print("SHAP executado com sucesso para o modelo de referencia: logistic_regression")
except Exception as e:
    print(f"SHAP nao pode ser executado neste ambiente: {e}")


SHAP executado com sucesso para o modelo de referencia: logistic_regression


C:\Users\ricoi\AppData\Local\Temp\ipykernel_21192\902260862.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
